# 33. Search in Rotated Sorted Array
**Difficulty:** 🟡 Medium · **Topic:** Array · **LeetCode:** https://leetcode.com/problems/search-in-rotated-sorted-array/

## 💡 Concepts

**Core concept(s):** **Modified binary search** — at each step, identify which half is properly sorted and decide whether the target lies inside it.

**Why it applies here:** After rotation, for any midpoint at least **one** side (`lo..mid` or `mid..hi`) is fully sorted. If the target falls within that sorted side's range, recurse there; otherwise recurse into the other side. Each step still halves the search space.

**Key intuition / mental model:** Find the sorted half, range-check the target against it, then keep the half that must contain it.

---

### 📚 What is Binary Search? (quick recap)
Halve a monotonic search space each step: inspect the middle, keep only the side that can hold the answer — **O(log n)** time, **O(1)** space. Here the twist is deciding *which* side is the trustworthy (sorted) one before comparing against the target.

## 📝 Problem

A sorted array of **distinct** integers is rotated at an unknown pivot. Return the index of `target`, or `-1`. Must run in **O(log n)**.

**Example**
```
Input:  nums = [4, 5, 6, 7, 0, 1, 2], target = 0    Output: 4
Input:  nums = [4, 5, 6, 7, 0, 1, 2], target = 3    Output: -1
```
**Constraints:** all values unique; `1 <= len(nums) <= 5000`.

> Two meaningfully distinct approaches: O(n) linear scan and the O(log n) binary search.

### Approach 1 — Linear Scan (worst)

**Idea:** Check each element for the target.

**Time complexity:** `O(n)`.

**Space complexity:** `O(1)`.

In [ ]:
from typing import List

def search_rotated_linear(nums: List[int], target: int) -> int:
    for i, x in enumerate(nums):           # scan every element...
        if x == target:                    # ...until we find the target
            return i
    return -1

### Approach 2 — Modified Binary Search (optimal)

**Idea:** Compute `mid`. Whichever side is sorted (detected by comparing endpoints to `nums[mid]`), test if `target` lies in that side's range; move into it if so, else into the other side.

**Time complexity:** `O(log n)`.

**Space complexity:** `O(1)`.

In [ ]:
from typing import List

def search_rotated_binary(nums: List[int], target: int) -> int:
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid                     # found it
        if nums[lo] <= nums[mid]:          # the LEFT half [lo..mid] is sorted
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1               # target lies in that sorted left half
            else:
                lo = mid + 1               # otherwise it's in the right half
        else:                              # the RIGHT half [mid..hi] is sorted
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1               # target lies in that sorted right half
            else:
                hi = mid - 1               # otherwise it's in the left half
    return -1

In [ ]:
# Correctness check
nums = [4, 5, 6, 7, 0, 1, 2]
for t in range(-1, 9):
    l, b = search_rotated_linear(nums, t), search_rotated_binary(nums, t)
    print(f"target={t:>2} -> linear={l:>2}, binary={b:>2}")
    assert l == b, "mismatch!"

# exhaustive: every rotation of 0..49, every target present or absent
base = list(range(50))
for k in range(50):
    rot = base[k:] + base[:k]
    for t in range(-2, 52):
        expected = rot.index(t) if t in rot else -1
        assert search_rotated_binary(rot, t) == expected
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |
| `O(n³)`       | ≈ **8×** |

Inputs are built to force the **worst case** (no early exit) so the measurement reflects the true bound. Sub-millisecond rows are noisy — look at the trend, not one number.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark   # shared: prints ratio table + optional log-log plot

def make_worst_case(n):
    base = list(range(n))
    nums = base[n // 2:] + base[:n // 2]   # rotated sorted array
    target = -1                            # absent -> linear scans the whole array
    return (nums, target)

solutions = {
    "linear O(n)    ": search_rotated_linear,
    "binary O(log n)": search_rotated_binary,
}
sizes = [2000, 4000, 8000, 16000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Find the sorted half first:** In a rotated array one half is always sorted; range-check the target there, then keep the correct half — preserves O(log n).
- **Endpoint comparison as the decision rule:** `nums[lo] <= nums[mid]` cleanly detects which side is sorted.
- **Signal to reach for it:** "rotated sorted array", "O(log n) search", "sorted with one discontinuity".
- **Related problems:** Find Minimum in Rotated Sorted Array, Search in Rotated Sorted Array II (with duplicates), Find Peak Element.
- **Common pitfalls:** (1) wrong boundary strictness in the range check (`<` vs `<=`); (2) forgetting the equal case `nums[lo] <= nums[mid]` for the sorted-left test; (3) duplicates break this exact form (that's the "II" variant).